# Паттерн 6: Swarm — рой взаимозаменяемых агентов

Swarm — децентрализованная модель координации, в которой нет фиксированного порядка и нет центрального диспетчера. Вместо этого равноправные агенты читают общий контекст, добавляют свой вклад и сами решают — передать работу коллеге или завершить. Каждый агент берёт информацию из общего состояния, обрабатывает её и через `Command(goto=...)` направляет управление следующему участнику роя.

Отличие от Handoffs: агенты взаимозаменяемы (одинаковый промпт), а не специализированы. Отличие от Round-Robin: нет фиксированного порядка — каждый агент сам выбирает следующего.

```mermaid
graph LR
    START((START)) --> alpha(["🔹 Alpha<br/><i>самоорганизуется</i>"])
    alpha -->|NEXT: beta| beta(["🔹 Beta<br/><i>самоорганизуется</i>"])
    alpha -->|NEXT: gamma| gamma(["🔹 Gamma<br/><i>самоорганизуется</i>"])
    alpha -->|DONE| finalizer(["📊 Finalizer<br/><i>собирает результат</i>"])
    beta -->|NEXT: alpha| alpha
    beta -->|NEXT: gamma| gamma
    beta -->|DONE| finalizer
    gamma -->|NEXT: alpha| alpha
    gamma -->|NEXT: beta| beta
    gamma -->|DONE| finalizer
    finalizer --> END((END))

    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class alpha,beta,gamma worker
    class finalizer aggregator
    class START,END terminal
```

In [1]:
import sys
sys.path.insert(0, "..")

import operator
import random
from typing import TypedDict, Annotated, Literal

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние роя

Состояние `SwarmState` содержит четыре поля. `task` — общая цель, которую рой должен раскрыть. `contributions` с редьюсером `operator.add` — список вкладов агентов: каждый агент добавляет свой результат, и LangGraph автоматически аккумулирует их. `iteration` — счётчик итераций для защиты от зацикливания. `final_result` — итоговый синтез, который формирует финализатор.

In [3]:
llm = get_llm()

class SwarmState(TypedDict):
    task: str                                          # Общая цель
    contributions: Annotated[list[str], operator.add]  # Вклады агентов (аддитивный)
    iteration: int                                     # Счётчик итераций
    final_result: str                                  # Итоговый синтез

## Промпт и список агентов

Все агенты в рое получают одинаковый промпт — это ключевое отличие от паттернов со специализированными агентами. Промпт инструктирует агента прочитать общий контекст (вклады предыдущих коллег), добавить новую информацию и в конце принять решение: передать работу другому агенту (`NEXT: имя`) или завершить (`DONE`). Список `AGENTS` определяет участников роя.

In [4]:
WORKER_PROMPT = """Ты — один из агентов в рое. Твоё имя: {name}.

Прочитай общий контекст (вклады предыдущих агентов) и добавь
НОВУЮ полезную информацию по задаче. Не повторяй написанное.

В конце ответа на ОТДЕЛЬНОЙ строке напиши ОДНО из:
- NEXT: другой_агент — если задача ещё не раскрыта и нужен другой агент
- DONE — если задача раскрыта достаточно

Доступные агенты: alpha, beta, gamma (кроме тебя)."""

AGENTS = ["alpha", "beta", "gamma"]

## Фабрика воркеров

Функция `make_worker` — фабрика, создающая узлы-воркеры для графа. Каждый воркер работает по одному алгоритму: собирает текущий контекст из `contributions`, отправляет LLM запрос с общим промптом, парсит ответ и принимает решение о маршрутизации.

Управляющая логика разделена на три ветки:
1. **Лимит итераций** (>= 5) — принудительное завершение, страховка от зацикливания.
2. **DONE** в последней строке ответа — агент считает задачу раскрытой и направляет к финализатору.
3. **NEXT: имя** — агент передаёт работу конкретному коллеге. Если парсинг не удался, срабатывает fallback с случайным выбором.

Маршрутизация реализована через `Command(goto=...)` — рёбра графа не нужны, LangGraph следует командам напрямую.

In [5]:
def make_worker(name: str):
    """Фабрика для создания воркеров роя."""
    others = [a for a in AGENTS if a != name]

    def worker(state: SwarmState) -> Command[Literal["alpha", "beta", "gamma", "finalizer"]]:
        context = "\n".join(state["contributions"]) if state["contributions"] else "Пока ничего нет."
        iteration = state["iteration"] + 1

        response = llm.invoke([
            SystemMessage(content=WORKER_PROMPT.format(name=name)),
            HumanMessage(content=f"Задача: {state['task']}\n\nОбщий контекст:\n{context}")
        ])

        content = response.content.strip()
        # Разделяем вклад и управляющую команду
        lines = content.split("\n")
        control_line = lines[-1].strip().upper()
        contribution = "\n".join(lines[:-1]).strip() or content

        print(f"  [{name}, итерация {iteration}] +{len(contribution)} символов")

        update = {
            "contributions": [f"[{name}]: {contribution}"],
            "iteration": iteration,
        }

        # Защита от зацикливания
        if iteration >= 5:
            print(f"  [{name}] Лимит итераций — завершаю")
            return Command(goto="finalizer", update=update)

        if "DONE" in control_line:
            print(f"  [{name}] Задача раскрыта — завершаю")
            return Command(goto="finalizer", update=update)

        # Парсим NEXT: agent_name
        if "NEXT:" in control_line:
            target = control_line.split("NEXT:")[-1].strip().lower()
            if target in others:
                print(f"  [{name}] → передаю {target}")
                return Command(goto=target, update=update)

        # Fallback: передать случайному другому
        target = random.choice(others)
        print(f"  [{name}] → передаю {target} (fallback)")
        return Command(goto=target, update=update)

    return worker

## Финализатор

Когда агент решает, что задача раскрыта (или достигнут лимит итераций), управление переходит к финализатору. Это не участник роя, а отдельный узел-синтезатор: он получает все вклады агентов, убирает повторы и формирует итоговый результат в 4-5 предложений.

In [6]:
def finalizer(state: SwarmState) -> dict:
    """Синтезирует итоговый результат из вкладов всех агентов."""
    all_contributions = "\n\n".join(state["contributions"])
    response = llm.invoke([
        SystemMessage(content=(
            "Синтезируй общий результат из вкладов нескольких агентов. "
            "Убери повторы, выдели ключевые выводы. 4-5 предложений."
        )),
        HumanMessage(content=f"Задача: {state['task']}\n\nВклады агентов:\n{all_contributions}")
    ])
    print(f"  [Финализатор] Синтез готов")
    return {"final_result": response.content}

## Сборка графа

Сборка графа минимальна: три узла-воркера, один финализатор и всего два явных ребра — `START → alpha` (точка входа) и `finalizer → END` (точка выхода). Все промежуточные переходы между воркерами управляются исключительно через `Command(goto=...)`, что делает граф полностью динамическим. Это ключевая особенность роя: маршруты не заданы заранее, а определяются агентами в реальном времени.

In [7]:
graph = StateGraph(SwarmState)

graph.add_node("alpha", make_worker("alpha"))
graph.add_node("beta", make_worker("beta"))
graph.add_node("gamma", make_worker("gamma"))
graph.add_node("finalizer", finalizer)

graph.add_edge(START, "alpha")  # Первый агент — alpha
# Рёбра между воркерами не нужны: Command(goto=...) управляет маршрутизацией
graph.add_edge("finalizer", END)

app = graph.compile()

## Запуск

Запускаем рой с задачей о преимуществах и рисках мультиагентных систем в production. Агент `alpha` начинает первым, читает пустой контекст и добавляет свой вклад. Затем он сам выбирает следующего участника. Процесс продолжается, пока один из агентов не решит, что задача раскрыта достаточно, или не сработает лимит в 5 итераций.

In [8]:
result = app.invoke({
    "task": "Преимущества и риски использования мультиагентных систем в production",
    "contributions": [],
    "iteration": 0,
    "final_result": "",
})

print(f"\n  Итого итераций: {result['iteration']}")
print(f"  Вкладов агентов: {len(result['contributions'])}")
print(f"\n  Результат:\n  {result['final_result'][:300]}...")

  [alpha, итерация 1] +2440 символов
  [alpha] → передаю beta


  [beta, итерация 2] +2016 символов
  [beta] → передаю gamma


  [gamma, итерация 3] +1555 символов
  [gamma] Задача раскрыта — завершаю


  [Финализатор] Синтез готов

  Итого итераций: 3
  Вкладов агентов: 3

  Результат:
  Мультиагентные системы в production дают заметные плюсы: они позволяют разделять ответственность, специализировать агентов под разные роли, выполнять подзадачи параллельно и повышать качество за счёт взаимной проверки. Особенно они полезны в сложных многошаговых сценариях, где одному агенту трудно о...


## Итоги

**Swarm** — децентрализованный паттерн, в котором равноправные агенты с одинаковым промптом по очереди дополняют общий контекст. Ключевые особенности:

- **Нет центрального диспетчера** — каждый агент сам решает, кому передать работу через `Command(goto=...)`.
- **Взаимозаменяемость** — все воркеры используют один промпт, в отличие от Handoffs с узкоспециализированными агентами.
- **Динамическая маршрутизация** — порядок обхода не фиксирован (в отличие от Round-Robin), а определяется агентами на лету.
- **Защита от зацикливания** — лимит итераций и `recursion_limit` гарантируют завершение.
- **Финализатор** — отдельный узел для синтеза результатов, который не является участником роя.